### ⚠️ Patched for Local Execution
This notebook was originally designed for Google Colab. It has been automatically patched:
- Google Colab-specific imports and `drive.mount()` calls have been commented out
- Colab file paths (`/content/drive/...`) have been replaced with relative paths (`./`)
- `!pip install` commands have been commented out (install packages in your venv instead)

**To run locally:** activate your Python virtual environment first, then run this notebook in VS Code or Jupyter.

In [ ]:
# [PATCHED] from google.colab import drive
# [PATCHED] drive.mount('./')

In [ ]:
import zipfile
import pandas as pd
import os

working_folder = './ Drive/TransformersCode/02-ECommerce/ProductRecommendations/'

zip_file_path = working_folder + 'flipkart_com-ecommerce_sample.zip'

with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
    zip_ref.extractall(working_folder)

csv_file_path = os.path.join(working_folder , 'flipkart_com-ecommerce_sample.csv')
df = pd.read_csv(csv_file_path)

df.head()

In [ ]:
df = df.astype(str)

df = df.map(lambda x: x.lower())
df.head()

In [ ]:
df['product_specifications'][0]

In [ ]:
import re

def extract_specifications(specifications):
    pairs = re.findall(r'"key"=>"(.*?)", "value"=>"(.*?)"', specifications)
    pairs_formatted = [f"{key}: {value}" for key, value in pairs]
    return ' '.join(pairs_formatted)

In [ ]:
df['product_specifications'] = df['product_specifications'].apply(extract_specifications)

In [ ]:
df = df.applymap(lambda x: re.sub('[^A-Za-z0-9]+', ' ', str(x)))

In [ ]:
df['combined_text'] = df['product_name'] + ' ' + df['description'] + ' ' + df['product_specifications']

In [ ]:
df.head()

In [ ]:
pip install -U sentence-transformers

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('all-mpnet-base-v2')

In [ ]:
sentence_embeddings = model.encode(df['combined_text'].tolist())
sentence_embeddings

In [ ]:
import numpy as np

embeddings_path = working_folder + "embeddings.npy"

np.save(embeddings_path, sentence_embeddings)